In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import Generators.BaseModel as bm

def triage(R):
    # Active mask
    diff_post = np.mean(R[-500:, :30], axis=0) - np.mean(R[-500:, 30:60], axis=0)
    act_mask  = np.count_nonzero(np.abs(diff_post) > 5, axis=0) > 0

    # Hallucination mask
    diff_pre   = np.mean(R[300:400, :30], axis=0) - np.mean(R[300:400, 30:60], axis=0)
    no_hall_mask  = np.count_nonzero( (np.abs(diff_pre) > 5) | (diff_post < -5), axis=0 ) == 0

    return R[:, :, act_mask & no_hall_mask]

D = bm.BuildModel()
region_idx = [D.occipital, D.parietal, D.temporal, D.frontal, D.prefrontal]
sel = D.area_list[D.prefrontal].tolist()

Jmax = 0.28
G = 0.46
DD = 0
# TODO run parallel
A = bm.BuildModel(J_max=Jmax, G=G)
A.runsim(trials=100, trial_length=20)
B = bm.BuildModel(J_max=Jmax, G=G, aging_mechanism={'spine','FI'}, age='A')
B.runsim(trials=100, trial_length=20)

# Diff = np.mean(triage(B.R),-1) - np.mean(triage(A.R),-1)
# # A.brain3d(triage(A.R))
# # B.brain3d(triage(B.R))
# # D.brain3d(np.stack([Diff]*5, axis=-1))   

diff  = np.mean(triage(A.R)[-500:, :30], axis=0) - np.mean(triage(A.R)[-500:, 30:60], axis=0)
fracsY = np.array([float(np.mean(np.abs(diff[idx]) > 5)) for idx in region_idx])
diff  = np.mean(triage(B.R)[-500:, :30], axis=0) - np.mean(triage(B.R)[-500:, 30:60], axis=0)
fracsA = np.array([float(np.mean(np.abs(diff[idx]) > 5)) for idx in region_idx])
D = np.abs(fracsY-fracsA)
DD = D if np.linalg.norm(D)> np.linalg.norm(DD) else DD
print(np.linalg.norm(D), np.linalg.norm(DD), DD)

# B.plot_neural_activity(selection=sel, rates = triage(B.R))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


C:\Users\david\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
C:\Users\david\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\numpy\_core\_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0
nan 0.0 0


In [103]:
%load_ext autoreload
%autoreload 2
import numpy as np
from joblib import Parallel, delayed
import Generators.BaseModel as bm

def triage(R):
    diff_post = np.mean(R[-500:, :30], axis=0) - np.mean(R[-500:, 30:60], axis=0)
    act_mask  = np.count_nonzero(np.abs(diff_post) > 5, axis=0) > 0
    diff_pre  = np.mean(R[300:400, :30], axis=0) - np.mean(R[300:400, 30:60], axis=0)
    no_hall   = np.count_nonzero((np.abs(diff_pre) > 5) | (diff_post < -5), axis=0) == 0
    return R[:, :, act_mask & no_hall]

def lobe_fracs(model, region_idx):
    Rt = triage(model.R)
    if Rt.shape[-1] == 0:
        return np.full(len(region_idx), np.nan)
    diff = np.mean(Rt[-500:, :30], axis=0) - np.mean(Rt[-500:, 30:60], axis=0)
    return np.array([float(np.mean(np.abs(diff[idx]) > 5)) for idx in region_idx])

def run_pair(Jmax, G, region_idx):
    A = bm.BuildModel(J_max=Jmax, G=G)
    A.runsim(trials=100, trial_length=20)
    fY = lobe_fracs(A, region_idx)
    del A  # R is large (~1.4 GB at float64); free before the next model

    B = bm.BuildModel(J_max=Jmax, G=G, aging_mechanism={'spine', 'FI'}, age='A')
    B.runsim(trials=100, trial_length=20)
    fA = lobe_fracs(B, region_idx)
    del B
    return Jmax, G, fY, fA

D = bm.BuildModel()
region_idx = [D.occipital, D.parietal, D.temporal, D.frontal, D.prefrontal]

j_maxes = [round(i * 0.01 + 0.21, 2) for i in range(31)]
Gs      = [round(j * 0.02 + 0.2, 2) for j in range(16)]
grid    = [(J, G) for J in j_maxes for G in Gs]

results = Parallel(n_jobs=-2, verbose=10)(   # tune n_jobs to RAM: ~3 GB per worker peak
    delayed(run_pair)(J, G, region_idx) for J, G in grid
)

# Reduce: largest per-lobe difference between Young and Aged
best_norm, best = -1, None
for J, G, fY, fA in results:
    d = np.abs(fY - fA)
    if np.isnan(d).any():
        continue
    n = np.linalg.norm(d)
    if n > best_norm:
        best_norm, best = n, (J, G, d)
print(best_norm, best)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


[Parallel(n_jobs=-2)]: Using backend LokyBackend with 11 concurrent workers.
[Parallel(n_jobs=-2)]: Done   3 tasks      | elapsed:   48.9s
[Parallel(n_jobs=-2)]: Done  10 tasks      | elapsed:   50.6s
[Parallel(n_jobs=-2)]: Done  19 tasks      | elapsed:  1.5min
[Parallel(n_jobs=-2)]: Done  28 tasks      | elapsed:  2.1min
[Parallel(n_jobs=-2)]: Done  39 tasks      | elapsed:  2.9min


KeyboardInterrupt: 